In [ ]:
import sys
sys.path.append("..")
import json

from assets.config import COLUMNS, FORMAT, RECORDBYTE
import struct
import pandas as pd
import os

In [2]:
chCodeDict = {
    "1": "NonCell",
    "2": "System Error",
    "3": "Defective Cell",
    "4": "Cell Check Error",
    "64": "Time Complete",
    "65": "Voltage Complete",
    "66": "Current Complete",
    "67": "Capacity Complete",
    "68": "OCV Step Complete",
    "69": "Last Step Complete",
    "70": "Stopped by User",
    "71": "Pause",
    "72": "Check Step Complete",
    "73": "Ended for Next Step",
    "74": "Impedance Step Completed",
    "75": "ADV Step Completed",
    "76": "LOOP Step Completed",
    "77": "Delta Voltage Completed",
    "78": "SOC Completed",
    "79": "Temperature Completed",
    "80": "Process completed(Time)",
    "81": "Process completed(Voltage)",
    "82": "Process completed(Current)",
    "83": "Process completed(Capacity)",
    "84": "Process completed(Watt)",
    "85": "Process completed(WattHour)",
    "86": "Standby(Temper.)",
    "87": "Short Test Completed",
    "90": "Watt Complete",
    "91": "WattHour Complete",
    "92": "Accumulated Capacity Complete",
    "93": "Accumulated WattHour Complete",
    "94": "Cycle Time Complete",
    "95": "CV Time Complete",
    "96": "Inspection Overvoltage",
    "97": "Inspection Undervoltage",
    "98": "Inspection Upper Limit Voltage",
    "99": "Inspection Lower Limit Voltage",
    "100": "Inspection Upper Limit Current",
    "101": "Inspection Lower Limit Current",
    "102": "Contact Failure 1",
    "103": "Contact Failure 2",
    "104": "Contact Failure 3",
    "105": "BadCell1",
    "106": "BadCell2",
    "107": "BadCell3",
    "108": "Short",
    "109": "Location Error (None)",
    "110": "Location Error ()",
    "111": "Drop Voltage Error",
    "128": "Voltage Upper Limit",
    "129": "Voltage Lower Limit",
    "130": "LimitI Pause",
    "131": "LimitI End",
    "134": "OCV Upper Limit",
    "135": "OCV Lower Limit",
    "136": "Current Upper Limit",
    "137": "Current Lower Limit",
    "138": "LimitV Pause",
    "139": "LimitV End",
    "142": "Capacity Upper Limit",
    "143": "Capacity Lower Limit",
    "152": "User Completion",
    "153": "Pause Completion",
    "154": "Next Step Completion",
    "165": "Temperature Upper Limit",
    "166": "Temperature Lower Limit",
    "167": "Pattern Info. Read Error",
    "168": "JIG Sensor Error",
    "169": "Chamber Error",
    "170": "Inverter Error",
    "171": "Equipment Stop",
    "172": "=+Compare Line Error",
    "173": "=-Compare Line Error",
    "174": "Compare Line Ch. Error",
    "175": "Voltage Compare Upper Limit",
    "176": "Current Compare Upper Limit",
    "177": "Temper. Compare Excess",
    "178": "Meter Compare Upper Limit",
    "179": "Meter Compare Lower Limit",
    "180": "JIG Error",
    "181": "Cut of Temper. Line Wire",
    "182": "Network Connection Error",
    "183": "Short Tester Error",
    "184": "Short Tester Connection Error",
    "186": "Short Tester Singleness Mode",
    "208": "Voltage Error (Warning)",
    "209": "Current Error (Warning)",
    "210": "Circuit Overheat (Warning)",
    "211": "Control Power Error (SMPS)",
    "212": "Power Off(Switch)",
    "213": "UPS Power Error",
    "214": "UPS Battery Error",
    "215": "EMG (MAIN)",
    "216": "EMG (SUB)",
    "217": "Network Error",
    "220": "AUX Temper. Completed",
    "221": "AUX Temper. Upper Limit Complet",
    "222": "AUX Temper. Lower Limit Complet",
    "223": "AUX Volt. Completed",
    "224": "AUX Volt. Upper Limit Completed",
    "225": "AUX Volt. Lower Limit Completed",
    "226": "AUX Temper. Upper Limit",
    "227": "AUX Temper. Lower Limit",
    "228": "AUX Volt. Upper Limit",
    "229": "AUX Volt. Lower Limit",
    "230": "Temper. Excess Completed",
    "231": "DC Fan Error",
    "232": "Capa. Efficiency Complete",
    "233": "Capa. Efficiency Pause",
    "234": "Capa. Efficiency Cycle Complete",
    "235": "Cycle Capacity Complete",
    "236": "Power Efficiency Cycle Complete",
    "237": "Imp. Efficiency Cycle Complete",
    "238": "Power Efficiency Pause",
    "239": "Imp. Efficiency Pause",
    "240": "Cycle Voltage Complete",
    "241": "Cycle Voltage End",
    "242": "Door Open Detected",
    "243": "Smoke Detected",
    "244": "Delta Voltage Error",
    "245": "Delta Current Error",
    "246": "Delta Voltage Complete",
    "247": "Delta Current Complete",
    "248": "Capa. Efficiency Complete",
    "249": "Power Efficiency Complete",
    "250": "Imp. Efficiency Complete",
    "251": "Pause(Grade Check)",
    "252": "Complete(Grade Check)",
    "253": "Channel Error in Same Chamber",
    "260": "[PCU]Input Voltage Error",
    "261": "[PCU]Voltage Upper Limit",
    "262": "[PCU]Current Upper Limit",
    "263": "[PCU]Temperature Upper Limit",
    "264": "[PCU]PV Voltage Error",
    "265": "[PCU]Terminal Voltage Error",
    "266": "[PCU]Output Current Error",
    "268": "[PCU]Dischar. Vol. Source Error",
    "269": "[PCU]Output Curr. Balance Error",
    "270": "[PCU]External Relay Error",
    "271": "[PCU]output Contact Error",
    "273": "[PCU]Communication Error",
    "274": "[PCU]Unknown Error",
    "275": "[PCU]Work Mode Error",
    "276": "[PCU]LAN Connect Error",
    "277": "[PCU]Sequence No. Error",
    "280": "[INV]LAG SHORT CURR.",
    "281": "[INV]OVER CURRENT",
    "282": "[INV]OVER VOLTAGE",
    "283": "[INV]PRECHARGE FAIL",
    "284": "[INV]OVER CURRENT 2",
    "285": "[INV]CAN ERROR",
    "286": "[INV]OVER LOAD",
    "287": "[INV]OVER HEAT",
    "289": "[INV]LOW VOLTAGE",
    "291": "[INV]RESET 1",
    "292": "[INV]RESET 2",
    "293": "[INV]AC INPUT FAIL",
    "295": "[INV]HDC ERROR",
    "296": "[INV]STAND-BY",
    "310": "CV Fault Voltage Error",
    "311": "CV Fault Current Error",
    "315": "Diagnosis Pause",
    "316": "Diagnosis Complete",
    "318": "3rd Current Calculation Anomaly",
    "360": "TVOC Upper",
    "361": "TVOC Lower",
    "362": "eCO2 Upper",
    "363": "eCO2 Lower",
    "364": "TVOC Complete",
    "365": "eCO2 Complete",
    "400": "Sample Vent Detect",
    "401": "Sample Vent Detect",
    "402": "Rest Voltage Error",
    "403": "Chamber SV Error",
    "404": "Temp upper(EMG)",
    "405": "Sample Soft Vent Detect",
    "406": "Sample Hard Vent Detect",
    "412": "Voltage upper(MonPro)",
    "413": "Voltage upper(OVP)",
    "416": "Temp upper(MonPro)",
    "417": "Temp upper(OTP)",
    "418": "Voltage lower(MonPro)",
    "419": "Pause by group(OVP)",
    "420": "Pause by group(OTP)",
    "421": "Chamber DI Error",
    "422": "Chamber SV, Ambient Temp Deviation Error",
    "423": "Group Voltage Error",
    "424": "Group Time Error",
    "426": "Chamber Stop",
    "500": "COMServer Communication Error",
    "1128": "Voltage upper(Step)",
    "1129": "Voltage lower(Step)",
    "1134": "Voltage upper(Step)",
    "1135": "Voltage lower(Step)",
    "1136": "Current upper(Step)",
    "1137": "Current lower(Step)",
    "1142": "Capacity upper(Step)",
    "1143": "Capacity lower(Step)",
    "1165": "Temp upper(Step)",
    "1166": "Temp lower(Step)",
    "2134": "Voltage upper(Test)",
    "2135": "Voltage lower(Test)",
    "2128": "Voltage upper(Test)",
    "2129": "Voltage lower(Test)",
    "2136": "Current upper(Test)",
    "2137": "Current lower(Test)",
    "2142": "Capacity upper(Test)",
    "2143": "Capacity lower(Test)",
    "2165": "Temp upper(Test)",
    "2166": "Temp lower(Test)"
}

### 4101(0x1005)

In [3]:
col_dict = {
    'StepNo': 'Step',
    'StepType': 'Type',
    'Mode': '상태',
    'TIME[HMS]': 'TIME[HMS]',
    'Voltage': '전압[V]',
    'Current': '전류[A]',
    'Temperature': '온도(℃)',
    'Capacity': '용량[Ah]',
    'ChargeCapacity': '충전용량(mAh)',
    'DischargeCapacity': '방전용량(mAh)',
    'Watt': 'Watt[W]',
    'WattHour': 'WattHour[Wh]',
    'ChargeWattHour': '충전WattHour[Wh]',
    'DischargeWattHour': '방전WattHour[Wh]',
    'AvgVoltage': '평균전압[V]',
    'AvgCurrent': '평균전류[A]',
    'Impedance': 'Imp[mΩ]',
    '총시간[HMS]': '총시간[HMS]',
    'TotalCycleNum': '총Cycle'
}

In [4]:
col_dict = {
    "ChCode":"Code",
    'Temperature': 'Temp',
    'Watt': 'Power',
    'TotalCycleNum': 'TotalCycle'
}

In [5]:
def read_ps_file_id_header(file):
    # Read and parse PS_FILE_ID_HEADER
    szFileID, szFileVersion = struct.unpack('@II', file.read(8))
    szCreateDateTime = file.read(64).decode('utf-8', errors='replace').rstrip('\x00')
    szDescription = file.read(128).decode('utf-8', errors='replace').rstrip('\x00')
    szReserved = file.read(128).decode('utf-8', errors='replace').rstrip('\x00')

    return {
        'szFileID': szFileID,
        'szFileVersion': szFileVersion,
        'szCreateDateTime': szCreateDateTime,
        'szDescription': szDescription,
        'szReserved': szReserved,
    }

In [7]:
def read_ps_test_file_header(file, version):
    record_bytes = int(RECORDBYTE[version])  # Get the record byte size for the given version
    column_mapping = {
        0x00: 'PS_STATE', 0x01: 'PS_VOLTAGE', 0x02: 'PS_CURRENT', 0x03: 'PS_CAPACITY',
        0x04: 'PS_IMPEDANCE', 0x05: 'PS_CODE', 0x06: 'PS_STEP_TIME', 0x07: 'PS_TOT_TIME',
        0x08: 'PS_GRADE_CODE', 0x09: 'PS_STEP_NO', 0x0A: 'PS_WATT', 0x0B: 'PS_WATT_HOUR',
        0x0C: 'PS_TEMPERATURE', 0x0D: 'PS_PRESSURE', 0x0E: 'PS_STEP_TYPE', 0x0F: 'PS_CUR_CYCLE',
        0x10: 'PS_TOT_CYCLE', 0x11: 'PS_TEST_NAME', 0x12: 'PS_SCHEDULE_NAME', 0x13: 'PS_CHANNEL_NO',
        0x14: 'PS_MODULE_NO', 0x15: 'PS_LOT_NO', 0x16: 'PS_DATA_SEQ', 0x17: 'PS_AVG_CURRENT',
        0x18: 'PS_AVG_VOLTAGE', 0x19: 'PS_CAPACITY_SUM', 0x1A: 'PS_CHARGE_CAP', 0x1B: 'PS_DISCHARGE_CAP',
        0x1C: 'PS_METER_DATA', 0x1D: 'PS_START_TIME', 0x1E: 'PS_END_TIME', 0x1F: 'PS_SHARING_INFO',
        0x20: 'PS_GOTO_COUNT', 0x21: 'PS_WATTHOUR_SUM', 0x22: 'PS_CHAR_WATTHOUR',
        0x23: 'PS_DISCHAR_WATTHOUR', 0x24: 'PS_INTEGRAL_CAPACITY', 0x25: 'PS_INTEGRAL_WATTHOUR',
        0x26: 'PS_CV_END_TIME', 0x27: 'PS_CYCLE_NUM', 0x28: 'PS_TOT_TIME_CARRY',
        0x29: 'PS_FARAD', 0x2A: 'PS_TEMPERATURE2', 0x2B: 'PS_DQDV', 0x2C: 'PS_CHARGE_CC_CAP',
        0x2D: 'PS_CHARGE_CV_CAP', 0x2E: 'PS_DISCHARGE_CC_CAP', 0x2F: 'PS_DISCHARGE_CV_CAP',
        0x30: 'PS_REALDATE', 0x31: 'PS_REALCLOCK', 0x32: 'PS_CHAMBER_TEMPERATURE',
        0x33: 'PS_AUX_TEMPERATURE', 0x34: 'PS_AUX_VOLTAGE', 0x3B: 'PS_AUX_THICKNESS2',
        0x3C: 'PS_AUX_PRESSURE1', 0x3D: 'PS_AUX_PRESSURE2', 0x3E: 'PS_AUX_PRESSURE3',
        0x3F: 'PS_AUX_PRESSURE4', 0x40: 'PS_AUX_THICKNESS1', 0x49: 'PS_AMBIENT_TEMP',
        0x4A: 'PS_GAS_VOLTAGE', 0x4B: 'PS_IMPEDANCE_100MS', 0x4C: 'PS_IMPEDANCE_1S',
        0x4D: 'PS_IMPEDANCE_5S', 0x4E: 'PS_IMPEDANCE_30S', 0x4F: 'PS_IMPEDANCE_60S',
        0x70: 'PS_RESTORED_FLAG' ,0x41: 'PS_GAS_CO2', 0x42:'PS_GAS_TEMP',
        0x43: 'PS_GAS_AH', 0x44: 'PS_GAS_BASELINE', 0x45: 'PS_GAS_TVOC',
        0x46: 'PS_GAS_ETHANOL', 0x47: 'PS_GAS_H2',
    }

    # Read PS_TEST_FILE_HEADER
    szStartTime = file.read(64).decode('utf-8',errors='replace').strip('\x00')
    szEndTime = file.read(64).decode('utf-8',errors='replace').strip('\x00')
    szSerial = file.read(64).decode('utf-8',errors='replace').strip('\x00')
    szUserID = file.read(32).decode('utf-8',errors='replace').strip('\x00')
    szDescript = file.read(128).decode('utf-8',errors='replace').strip('\x00')
    szTrayNo = file.read(64).decode('utf-8',errors='replace').strip('\x00')
    szBuff = file.read(64).decode('utf-8',errors='replace').strip('\x00')
    
    nRecordSize = struct.unpack('@i', file.read(4))[0]
    wRecordItem = struct.unpack(f'@{record_bytes}H', file.read(record_bytes * 2))
    column_names = [column_mapping.get(item, f'UNKNOWN_0x{item:02X}') for item in wRecordItem[:nRecordSize]]
    awColumnItem_dict = dict(zip(column_names, wRecordItem[:nRecordSize]))

    return {
        'szStartTime': [szStartTime],
        'szEndTime': [szEndTime],
        'szSerial': [szSerial],
        'szUserID': [szUserID],
        'szDescript': [szDescript],
        'szTrayNo': [szTrayNo],
        'szBuff': [szBuff],
        'nRecordSize': [nRecordSize],
        'wRecordItem' : [wRecordItem],
        'awColumnItem': awColumnItem_dict,
        'columnNames': column_names
    }
    
def read_record_data(file, version):
    # Read the record data
    data = []
    format_string = "@" + FORMAT[version]
    record_size = struct.calcsize(format_string)
    print(record_size)
    while True:
        record = file.read(record_size)  # 180바이트 읽기
        if not record:
            break
        if len(record) == record_size:
            awColumnItem = struct.unpack(format_string, record)
            data.append(awColumnItem)
        else:
         # 예상보다 적은 바이트가 읽혔을 경우 (파일 끝 부분의 불완전한 레코드)
            print(f"Incomplete record found at EOF. Expected {record_size} bytes, got {len(record)} bytes.")
            break      
    
    return data

def convert_nTime(df):
    # "공정 시간"을 초 단위로 timedelta 변환 (100으로 나눠서 초로 변환)
    df['cumtime_td'] = pd.to_timedelta(df['TotalTime'], unit='s')
    # 누적 시간 → "D:HH:MM:SS" 변환
    df['TotalTime'] = df['cumtime_td'].apply(lambda x: f"{x.days}:{x.components.hours:02}:{x.components.minutes:02}:{x.components.seconds:02}")
    # 개별 공정 시간 계산 (현재 행 - 이전 행, 첫 번째 행은 그대로 사용)
    df['time'] = df['cumtime_td'].diff().fillna(df['cumtime_td'])
    # 개별 공정 시간 → "HH:MM:SS" 변환
    df['StepTime'] = df['time'].apply(lambda x: f"{x.components.hours:02}:{x.components.minutes:02}:{x.components.seconds:02}")
    # 필요 없는 timedelta 컬럼 삭제
    df = df.drop(columns=['cumtime_td', 'time'])
    
    return df

def preprocess_data(data: list, version: str)-> pd.DataFrame:
    column_names = COLUMNS[version]
    filtered_data = [record for record in data]

    # Convert filtered data to DataFrame
    df = pd.DataFrame(filtered_data, columns=column_names)
    # Divide PS_VOLTAGE, PS_CAPACITY, and PS_CURRENT by 1000 if they exist in the DataFrame
    for column in ['Voltage', 'Capacity', 'Current', 'Watt', 'WattHour','ChargeCapacity', 'DischargeCapacity', 'AvgVoltage', 'AvgCurrent']:
        if column in df.columns:
            df[column] = df[column] / 1000

    # Set PS_WATT to negative if PS_CURRENT is negative
    if 'Current' in df.columns and 'Watt' in df.columns:
        df.loc[df['Current'] < 0, 'Watt'] = df['Watt'].abs() * -1

    df = convert_nTime(df) # 시간 변환
    df['StepType'] = df['StepType'].replace({0:"None",1:"Charge", 2:"Discharge", 3:"Rest",4:"OCV",5:"IMPEDANCE", 6:"Completed", 7:"ADV_Cycle", 8:"Loop", 9:"Pattern", 10:"Balance", 11:"USERMAP"})
    df['Mode'] = df['Mode'].replace({0:"None", 1:"CCCV", 2:"CC", 3:"CV", 4:"DCIMP", 5:"ACIMP", 6:"CP" , 7:"PUSE", 8:"CR"})
    df['ChCode'] = df['ChCode'].map(lambda x: chCodeDict.get(str(x)))
    return df

def process_rpt(stepend, cell_type):
    if cell_type == 'JF1':
        criteria_current = 14.5
    elif cell_type == 'JF1R':
        criteria_current = 8.625
    elif cell_type == 'JF2':
        criteria_current = 39.80
    elif cell_type == 'JF2S':
        criteria_current = 38.950
    elif cell_type == 'JF3':
        criteria_current = 43.275
    else:
        criteria_current = None
        
    stepend['TestMode'] = 'In-situ'
    stepend['IsRptCapa'] = False
    stepend['IsRptDcir'] = False
    
    temp = stepend['StepNo'].value_counts()
    rpt_temp = temp[temp < temp.values.mean()].sort_index()

    stepend_rpt = stepend[(stepend['StepNo'].isin(rpt_temp.index))]
    stepend.loc[stepend_rpt.index, 'TestMode'] = 'RPT'

    def cluster_numbers_by_difference(numbers, threshold=10):
        if not numbers:
            return []

        clusters = []
        current_cluster = [numbers[0]]

        for i in range(1, len(numbers)):
            if numbers[i] - numbers[i-1] > threshold:
                clusters.append(current_cluster)
                current_cluster = [numbers[i]]
            else:
                current_cluster.append(numbers[i])

        clusters.append(current_cluster)

        return clusters

    rpt_group = cluster_numbers_by_difference(stepend_rpt.index.tolist())  #rpt 별로 group 을 만들어준다. 그룹 내에서 capa가 제일 큰걸로 rpt 용량인지 판별하기 때문에
    rpt_capa_idx, rpt_dcir_idx = [], []
    
    for mini_group in rpt_group:
        rpt_df = stepend.loc[mini_group]
        rpt_df = rpt_df[(rpt_df['StepType'] == 'Discharge') | (rpt_df['StepType'] == 'Charge')]
        if rpt_df.empty:
            continue
        if criteria_current is not None:
            condition = (rpt_df["Current"].abs() >= criteria_current - 2) & \
                    (rpt_df["Current"].abs() <= criteria_current + 2) & (rpt_df['StepType'] == 'Discharge')

            rpt_idx_discharge_df = rpt_df[condition]
            print(rpt_idx_discharge_df)
            if len(rpt_idx_discharge_df) != 0:
                if len(rpt_idx_discharge_df) >= 2:
                    rpt_idx_discharge = rpt_idx_discharge_df["DischargeCapacity"].idxmax()
                else:
                    rpt_idx_discharge = rpt_idx_discharge_df.index[0]
            else:
                rpt_idx_discharge = rpt_df["DischargeCapacity"].idxmax() 
            
        else:
            rpt_idx_discharge = rpt_df["DischargeCapacity"].idxmax() 

        charge_df = rpt_df.loc[:rpt_idx_discharge].query("StepType == 'Charge'")
        rpt_idx_charge = charge_df.index.max() if not charge_df.empty else None
        
        print(f"rpt_idx_charge : {rpt_idx_charge} / rpt_idx_discharge : {rpt_idx_discharge} ")
        print(rpt_df[['StepType', 'Current', 'Voltage', 'ChargeCapacity', 'DischargeCapacity']])
        
        if rpt_idx_charge is not None:
            rpt_capa_idx.append(rpt_idx_charge)
        if rpt_idx_discharge is not None:
            rpt_capa_idx.append(rpt_idx_discharge)

    stepend.loc[rpt_capa_idx, 'IsRptCapa'] = True
    stepend.loc[(stepend["StepType"] == "Discharge") & (stepend["Code"] == "Time Complete"), "IsRptDcir"] = True
    
    return stepend 
    
def parse_binary_file(file_path):
    with open(file_path, 'rb') as file:
        file_id_header = read_ps_file_id_header(file) # Read headers
        firmware_version = str(file_id_header['szFileVersion'])
        print(file_id_header)
        print(firmware_version)
        if firmware_version == '4101' and str(file_id_header['szFileID']) == '20200701': # 4101인데 Gas로 해야 맞음... 헤더에는 Gas를 식별할 수 있는 게 안붙어있어서 걍 이렇게 가정.
            firmware_version = '4101_Gas'
        record_file_header = read_ps_test_file_header(file, firmware_version) # Read record data
        current_position = file.tell()
        
        # 2. 포인터를 파일 끝으로 이동하여 전체 크기 확인
        file.seek(0, os.SEEK_END) 
        total_size = file.tell()
        
        # 3. 포인터를 원래 위치로 복원
        file.seek(current_position) 
        
        # 4. 남은 바이트 계산
        remaining_bytes = total_size - current_position
        
        global record_data
        record_data = read_record_data(file, firmware_version) # Read record data
        print(record_data)
        processed_data = preprocess_data(record_data, firmware_version)
        processed_data = processed_data.rename(columns=col_dict)
        processed_data = process_rpt(processed_data, 'JF2')
        
        return {
            'FileIDHeader': file_id_header,
            'RecordFileHeader': record_file_header,
            'RecordData' : record_data,
            'StepEndData' : processed_data
        }

file_path = r'C:\Users\jhchoei\Desktop\Workspace\VoltaFlow\data\error 확인용\07001967_20260127_JF2S_DV2_T25_03_075CP_UEDQF23859_No2\M01Ch008[008]\07001967_20260127_JF2S_DV2_T25_03_075CP_UEDQF23859_No2.cts'
cts_parse = parse_binary_file(file_path)

{'szFileID': 20240905, 'szFileVersion': 8196, 'szCreateDateTime': '2026-02-09 ���� 4:29:00', 'szDescription': 'PNE Cycler step end data file.', 'szReserved': ''}
8196
232
[(8, 2, 2, 3, 1, 0, 0, 0, 0, 3600, 1, 1, 3601, 0, 3305.470947265625, -10.942000389099121, 0.0, 0.0, 0.0, 3600.0, 3600.0, 0.0, 24.562999725341797, 0.0, 3305.39990234375, -10.800000190734863, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, b'\x00', b'\x00', 1, 25.0, 0.0, 0.0, 0.0, 0.0, 64, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 260127.0, 112554728.0, 25.0, 999.0, 999.0, 999.0, 999.0, -11728.0, -11726.0, 0.0, 0.0, 5.775000095367432, 3.2799999713897705, 0.0, 0.0), (8, 5, 2, 2, 1, 0, 0, 2, 3601, 27542, 1, 2, 27543, 0, 2499.988037109375, -15578.8515625, 100519.4296875, 38946.0, 322355.0, 23228.169921875, 26828.169921875, 0.8479999899864197, 24.562999725341797, 0.0, 3207.10009765625, -15579.0, 0, 0.0, 0.0, 0.0, 100519.4296875, 0.0, 322.3550109863281, 0.0, 0.0, b'\x00', b'\x00', 2, 25.0, 0.0, 0.0, 0.0, 0.0, 65, 0, 0.0, 0.0, 0.0, 0.

In [16]:
cts_parse['StepEndData']

,No,StepNo,State,StepType,DataSelect,Reserved1,GradeCode,Mode,IndexFrom,IndexTo,...,fAuxPressure[4]2,fAuxPressure[4]3,fAuxPressure[4]4,fAuxThickness[4]1,fAuxThickness[4]2,fAuxThickness[4]3,fAuxThickness[4]4,TestMode,IsRptCapa,IsRptDcir
0,0,0,14,Discharge,2,3,1,None,0,235929600,...,-2.423406e-41,2.423546e-41,0.0,3.799861e-24,-7.892611e+17,2.306537e-41,0.0,In-situ,False,False
1,0,0,14,IMPEDANCE,2,2,1,None,235995648,2239102976,...,-2.422985e-41,2.423125e-41,0.0,1.409717e-10,1.268494e-26,2.306818e-41,0.0,In-situ,False,False
2,0,0,14,Completed,2,3,1,None,2239168512,2405761024,...,-2.422985e-41,2.423125e-41,0.0,1.948176e-20,1.268494e-26,2.306818e-41,0.0,In-situ,False,False
3,0,0,14,ADV_Cycle,2,1,1,None,2405826816,3427467264,...,-2.425648e-41,2.425788e-41,0.0,-2.656420e+15,-5.012776e-05,2.305696e-41,0.0,In-situ,False,False
4,0,0,14,Loop,2,3,1,None,3427532800,3594911744,...,2.426068e-41,2.426208e-41,0.0,9.882812e-01,-7.162553e+01,2.305556e-41,0.0,In-situ,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,0,0,14,25,2,1,1,None,1281754624,2131558421,...,2.425227e-41,2.356423e-41,0.0,1.902771e+17,2.611624e+07,2.305837e-41,0.0,In-situ,False,False
240,0,0,14,26,2,3,1,None,2131623936,2220621845,...,2.425087e-41,2.353060e-41,0.0,-9.779726e+28,-9.795268e+28,2.305837e-41,0.0,In-situ,False,False
241,0,0,14,27,2,2,1,None,2220688896,2571698197,...,2.425508e-41,2.360908e-41,0.0,-1.881391e+25,-9.128136e-25,2.305696e-41,0.0,In-situ,False,False
242,0,0,14,28,2,3,1,None,2571763712,2819489813,...,2.424527e-41,2.340729e-41,0.0,-4.943837e+32,6.344230e+14,2.306117e-41,0.0,In-situ,False,False


In [10]:
df = cts_parse['StepEndData']

In [13]:
df['Code']

0          Time Complete
1       Voltage Complete
2          Time Complete
3       Current Complete
4          Time Complete
              ...       
1576       Time Complete
1577    Voltage Complete
1578       Time Complete
1579    Voltage Complete
1580       Time Complete
Name: Code, Length: 1581, dtype: object

In [58]:
stepend_data["Cell_ID"] = "DA05B1R907"
stepend_data["EXP_ID"] = "EXP_241113_00785"

In [59]:
import psycopg2
from io import StringIO

try:
    conn = psycopg2.connect(host="10.99.212.29",
                            database="mart_db",
                            user="postgres",
                            password="lifemodeling",
                            port=5432,
                            client_encoding='utf-8') # 이 부분을 추가
    print("데이터베이스 연결 성공!")
    # 여기에 데이터베이스 작업 코드 추가
    cur = conn.cursor()
    csv_buffer = StringIO()
    temp = stepend_data[['Cell_ID', 'EXP_ID', 'StepNo', 'StepType', 'Code','TotalCycle', 'Voltage', 'Current',
                        'Capacity', 'Power', 'WattHour', 'StepTime', 'TotalTime', 'Impedance', 'Temp',
                        'AvgVoltage', 'AvgCurrent', 'ChargeCapacity', 'DischargeCapacity', 'ChargeWattHour',
                        'DischargeWattHour', 'CVEndTime', 'TestMode', 'IsRptCapa', 'IsRptDcir']]
    temp.to_csv(csv_buffer, index=False, header=False, sep='\t', encoding='utf-8')
    csv_buffer.seek(0)  # StringIO 버퍼의 시작으로 이동
    cur.copy_from(csv_buffer, 'exp_fact_tb', sep='\t', columns=['cell_id', 'exp_id', 'stepno', 'steptype', 'code', 'totalcycle', 'voltage', 'current', 'capacity', 'power', 'watthour', 'steptime', 'totaltime', 'impedance', 'temp', 'avgvoltage', 'avgcurrent', 'chargecapacity', 'dischargecapacity', 'chargewatthour', 'dischargewatthour', 'cvendtime', 'testmode', 'isrptcapa', 'isrptdcir'])
    conn.commit()  # 변경 사항을 커밋합니다.
    # cur.execute("SELECT * FROM exp_fact_tb LIMIT 0")  # 테이블 구조 확인
    # # --- 여기서부터 컬럼명을 확인하는 코드 ---
    # if cur.description:
    #     column_names = [col[0] for col in cur.description]
    #     print("테이블 컬럼명:", column_names)
    # else:
    #     print("컬럼 정보를 가져올 수 없습니다. 쿼리가 올바른지 확인해주세요.")
    # --- 여기까지 컬럼명을 확인하는 코드 ---
    #cur.execute("INSERT INTO exp_fact_tb(CELL_ID, EXP_ID) VALUES ('ASD', 'SBS')")
    #conn.commit()

except Exception as e:
    print(f"데이터베이스 오류 발생: {e}")

finally:
    if cur:
        cur.close()
    if conn:
        conn.close()

데이터베이스 연결 성공!


In [ ]:
cts_parse['StepEndData'].columns

Index(['No', 'StepNo', 'State', 'StepType', 'DataSelect', 'Code', 'IndexFrom',
       'IndexTo', 'CurrentCycleNum', 'TotalCycle', 'SaveSequence', 'Voltage',
       'Current', 'Capacity', 'Power', 'WattHour', 'StepTime', 'TotalTime',
       'Impedance', 'Temp', 'AvgVoltage', 'AvgCurrent', 'ChargeCapacity',
       'DischargeCapacity', 'ChargeWattHour', 'DischargeWattHour', 'CVEndTime',
       'TotalTimeCarry', 'Reserved3', 'CycleNum', 'ChamberTemp',
       'ChargeCCCapacity', 'ChargeCVCapacity', 'DischargeCCCapacity', 'ChCode',
       'Impedance100ms', 'Impedance1s', 'Impedance5s', 'Impedance30s',
       'Impedance60s', 'RealDate', 'RealClock', 'TestMode', 'IsRptCapa',
       'IsRptDcir'],
      dtype='object')

In [7]:
cts_parse['StepEndData']["TotalTime"]

0         0:04:00:00
1         0:04:02:56
2         0:04:32:56
3         0:08:30:39
4         0:08:50:39
            ...     
3638    112:14:21:38
3639    112:18:14:44
3640    112:18:34:44
3641    112:22:18:01
3642    112:23:18:01
Name: TotalTime, Length: 3643, dtype: object

In [6]:
from utils.excel_automation import read_template, insert_data

from datetime import datetime
import pandas as pd
import xlwings as xw
import os
import sys

app = xw.App(visible=True)  # 엑셀 창이 안 보이도록 설정
wb = app.books.open(r"C:\Users\jhchoei\Desktop\Workspace\TOS\assets\template.xlsx")  # app을 명시적으로 사용
sheet = wb.sheets["template"]

now = datetime.now().strftime('%Y-%m-%d')
sheet.range('A2').value = now

insert_data(cts_parse['StepEndData'], sheet, "dsdsd", "dss")

In [ ]:
import sys
sys.path.append(r"C:\Users\jhchoei\Desktop\Workspace\TOS")
from processor.PNE import CtsProcessor
from assets import config_prevdd

processor = CtsProcessor(config_prevdd)
results = processor.parse_binary_file(file_path)


In [ ]:
processor = CtsProcessor(config_prevdd)
results = processor.parse_binary_file(file_path)